In [1]:
import random
import torch
import torch.nn as nn
from copy import deepcopy
from collections import deque
from blackjack import Environment
from utils import get_game_state, get_game_state_sparse

In [2]:
class Player(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(Player, self).__init__()

        self.stack_1 = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU()
        )

        self.stack_2 = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Linear(hidden_size // 2, hidden_size // 2),
            nn.ReLU()
        )

        self.output = nn.Linear(hidden_size // 2, 2)

    def forward(self, x):
        x = self.stack_1(x)
        x = self.stack_2(x)

        return self.output(x)

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
learning_rate = 5e-4
model = Player(input_size=6, hidden_size=128).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.99)

In [4]:
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable_params:,}")

Trainable parameters: 29,954


In [5]:
target_network = deepcopy(model)  # target network for stability during learning
memory = deque(maxlen=25_000)  # memory buffer for experience replay
batch_size = 128
gamma = 0.99  # bellman equation constant
epsilon = 1.0  # base exploration chance

In [7]:
def choose_action(e, state) -> int:
    """
    Choose an action according to the epsilon-greedy policy. 0 for hit, 1 for stay
    ---------------------------
    Parameters:
        e (float): the exploration chance
        state (torch.tensor): the current state of the game
    """
    if random.random() < e:
        return random.choice([0, 1])

    state = state.to(device)
    output = model(state)
    return torch.argmax(output).item()


def train():
    if len(memory) < batch_size:
        return

    batch = random.sample(memory, batch_size)
    states, actions, rewards, dones, next_states = zip(*batch)

    states = torch.stack(states).to(device)
    actions = torch.tensor(actions, dtype=torch.int64, device=device).reshape(batch_size)
    rewards = torch.tensor(rewards, dtype=torch.float32).reshape(batch_size).to(device)  # Convert rewards to a tensor
    dones = torch.tensor(dones, dtype=torch.float32).reshape(batch_size).to(device)  # Convert dones to a tensor

    q_values = model(states)
    predicted_q_values = q_values.gather(1, actions.unsqueeze(1)).squeeze(1)  # get the q values for the corresponding actions taken

    next_q_values = torch.zeros(batch_size, device=device)

    non_terminal_mask = (dones == 0)  # Mask for non-terminal states
    non_terminal_next_states = [s for s in next_states if s is not None]

    if non_terminal_next_states:
        non_terminal_next_states = torch.stack(non_terminal_next_states).to(device)
        next_q_values[non_terminal_mask] = torch.max(target_network(non_terminal_next_states), dim=1).values

    target_q_values = rewards + gamma * next_q_values * (1 - dones)  # target q values according to bellman equation

    loss = criterion(target_q_values, predicted_q_values)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

recent_wins = 0
for episode in range(1, 200_001):
    if episode % 100 == 0:
        epsilon = max(epsilon * 0.99, 0.05)  # decay the exploration chance

    env = Environment(five_card_charlie=True)  # each episode starts a new game; the deck is NOT re-used.
    while not env.is_over:
        state = get_game_state(env)
        action = choose_action(epsilon, state)

        if action == 0:
            env.step("H")
        else:
            env.step("S")

        reward = env.score
        done = int(env.is_over)

        if done:
            next_state = None
        else:
            next_state = get_game_state(env)

        memory.append((state, action, reward, done, next_state))

    if env.score == 1:
        recent_wins += 1

    if episode % 4:
        train()

    if episode % 1000 == 0:
        target_network.load_state_dict(model.state_dict())
        scheduler.step()

    if episode % 5000 == 0:
        print(f"Episode: {episode}, Winrate in last 5k games: {recent_wins / 5000:.4f}")
        recent_wins = 0

Episode: 5000, Winrate in last 5k games: 0.3394
Episode: 10000, Winrate in last 5k games: 0.3582
Episode: 15000, Winrate in last 5k games: 0.3724
Episode: 20000, Winrate in last 5k games: 0.3808
Episode: 25000, Winrate in last 5k games: 0.3888
Episode: 30000, Winrate in last 5k games: 0.3886
Episode: 35000, Winrate in last 5k games: 0.4080
Episode: 40000, Winrate in last 5k games: 0.4028
Episode: 45000, Winrate in last 5k games: 0.4064
Episode: 50000, Winrate in last 5k games: 0.3990
Episode: 55000, Winrate in last 5k games: 0.4020
Episode: 60000, Winrate in last 5k games: 0.3966
Episode: 65000, Winrate in last 5k games: 0.3992
Episode: 70000, Winrate in last 5k games: 0.4004
Episode: 75000, Winrate in last 5k games: 0.4174
Episode: 80000, Winrate in last 5k games: 0.4092
Episode: 85000, Winrate in last 5k games: 0.4124
Episode: 90000, Winrate in last 5k games: 0.3990
Episode: 95000, Winrate in last 5k games: 0.4042
Episode: 100000, Winrate in last 5k games: 0.4068
Episode: 105000, Win

In [8]:
torch.save(model.state_dict(), "bj_player_dense.pt")

In [9]:
model.load_state_dict(torch.load("bj_player_dense.pt", weights_only=True))

<All keys matched successfully>

In [10]:
model.eval()
wins = 0
num_games = 10_000
for _ in range(num_games):
    env = Environment(five_card_charlie=True)
    # print("=============================")
    # print("Player hand:", *env.player_hand, "\nDealer hand:", *env.dealer_hand)
    while not env.is_over:
        state = get_game_state(env).to(device)
        output = model(state)
        # print(output)
        action = torch.argmax(output).item()
        if action == 0:
            # print("Bot hits")
            env.step("H")
        else:
            # print("Bot stays")
            env.step("S")
        # print("-----------------------------")
        # print("Player hand:", *env.player_hand, "\nDealer hand:", *env.dealer_hand)

    # print("Score:", env.score)
    wins += 1 if env.score == 1 else 0

print(f"Win rate: {100 * wins / num_games:.4f}%")

Win rate: 41.5200%
